# Emission Prediction Model

This notebook predicts CO2 emissions for shipments in the Green Supply Chain Tracker system using machine learning models.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


## 1. Load and Explore Data


In [ ]:
# Generate sample shipment data based on the project's data structure
np.random.seed(42)

# Vehicle types and fuel types from the project
vehicle_types = ['truck', 'train', 'ship', 'plane', 'van']
fuel_types = ['diesel', 'petrol', 'electric', 'hybrid', 'cng']

# Emission factors (from shipment.ts)
emission_factors = {
    ('truck', 'diesel'): 0.62,
    ('truck', 'petrol'): 0.68,
    ('truck', 'electric'): 0.15,
    ('truck', 'hybrid'): 0.35,
    ('truck', 'cng'): 0.45,
    ('van', 'diesel'): 0.48,
    ('van', 'petrol'): 0.52,
    ('van', 'electric'): 0.12,
    ('train', 'diesel'): 0.22,
    ('train', 'electric'): 0.05,
    ('ship', 'diesel'): 0.015,
    ('plane', 'petrol'): 1.2,
}

# Generate synthetic data
n_samples = 1000
data = []

for i in range(n_samples):
    vehicle = np.random.choice(vehicle_types)
    # Select appropriate fuel type for vehicle
    if vehicle == 'truck':
        fuel = np.random.choice(['diesel', 'petrol', 'electric', 'hybrid', 'cng'])
    elif vehicle == 'van':
        fuel = np.random.choice(['diesel', 'petrol', 'electric'])
    elif vehicle == 'train':
        fuel = np.random.choice(['diesel', 'electric'])
    elif vehicle == 'ship':
        fuel = 'diesel'
    else:  # plane
        fuel = 'petrol'
    
    weight = np.random.uniform(100, 5000)  # kg
    distance = np.random.uniform(50, 2000)  # km
    
    # Calculate actual emissions with some noise
    factor = emission_factors.get((vehicle, fuel), 0.5)
    base_emissions = factor * distance * (weight / 1000)
    noise = np.random.normal(0, base_emissions * 0.1)  # 10% noise
    emissions = max(0, base_emissions + noise)
    
    data.append({
        'vehicle_type': vehicle,
        'fuel_type': fuel,
        'weight': weight,
        'distance': distance,
        'emissions_co2': emissions
    })

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
df.head()


In [ ]:
# Data summary
print("Dataset Statistics:")
print(df.describe())
print("\nVehicle Type Distribution:")
print(df['vehicle_type'].value_counts())
print("\nFuel Type Distribution:")
print(df['fuel_type'].value_counts())


## 2. Data Visualization


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Distribution of emissions
axes[0, 0].hist(df['emissions_co2'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of CO2 Emissions', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('CO2 Emissions (kg)')
axes[0, 0].set_ylabel('Frequency')

# Emissions by vehicle type
vehicle_emissions = df.groupby('vehicle_type')['emissions_co2'].mean().sort_values(ascending=False)
axes[0, 1].bar(vehicle_emissions.index, vehicle_emissions.values, color='coral')
axes[0, 1].set_title('Average Emissions by Vehicle Type', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Vehicle Type')
axes[0, 1].set_ylabel('Average CO2 Emissions (kg)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Emissions by fuel type
fuel_emissions = df.groupby('fuel_type')['emissions_co2'].mean().sort_values(ascending=False)
axes[1, 0].bar(fuel_emissions.index, fuel_emissions.values, color='lightblue')
axes[1, 0].set_title('Average Emissions by Fuel Type', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Fuel Type')
axes[1, 0].set_ylabel('Average CO2 Emissions (kg)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Scatter: Distance vs Emissions
axes[1, 1].scatter(df['distance'], df['emissions_co2'], alpha=0.5, s=20)
axes[1, 1].set_title('Distance vs CO2 Emissions', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Distance (km)')
axes[1, 1].set_ylabel('CO2 Emissions (kg)')

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## 3. Feature Engineering


In [ ]:
# Encode categorical variables
le_vehicle = LabelEncoder()
le_fuel = LabelEncoder()

df_encoded = df.copy()
df_encoded['vehicle_type_encoded'] = le_vehicle.fit_transform(df['vehicle_type'])
df_encoded['fuel_type_encoded'] = le_fuel.fit_transform(df['fuel_type'])

# Create additional features
df_encoded['weight_ton'] = df_encoded['weight'] / 1000
df_encoded['distance_weight_ratio'] = df_encoded['distance'] / df_encoded['weight']
df_encoded['emission_intensity'] = df_encoded['emissions_co2'] / df_encoded['distance']

# Prepare features for modeling
feature_cols = ['vehicle_type_encoded', 'fuel_type_encoded', 'weight', 'distance', 
                'weight_ton', 'distance_weight_ratio']
X = df_encoded[feature_cols]
y = df_encoded['emissions_co2']

print("Feature columns:", feature_cols)
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")


## 4. Model Training


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    results[name] = {
        'model': model,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'y_test_pred': y_test_pred
    }
    
    print(f"  Train MAE: {train_mae:.2f} kg")
    print(f"  Test MAE: {test_mae:.2f} kg")
    print(f"  Train RMSE: {train_rmse:.2f} kg")
    print(f"  Test RMSE: {test_rmse:.2f} kg")
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²: {test_r2:.4f}")


## 5. Model Comparison


In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test MAE (kg)': [results[m]['test_mae'] for m in results.keys()],
    'Test RMSE (kg)': [results[m]['test_rmse'] for m in results.keys()],
    'Test R²': [results[m]['test_r2'] for m in results.keys()]
})

comparison_df = comparison_df.sort_values('Test R²', ascending=False)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))


In [ ]:
# Visualize model performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, result) in enumerate(results.items()):
    y_pred = result['y_test_pred']
    
    axes[idx].scatter(y_test, y_pred, alpha=0.5, s=30)
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                   'r--', lw=2, label='Perfect Prediction')
    axes[idx].set_xlabel('Actual Emissions (kg)', fontsize=12)
    axes[idx].set_ylabel('Predicted Emissions (kg)', fontsize=12)
    axes[idx].set_title(f'{name}\nR² = {result["test_r2"]:.4f}', fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Feature Importance


In [ ]:
# Feature importance for Random Forest
best_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance (Random Forest)', fontsize=16, fontweight='bold')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))


## 7. Predictions for New Shipments


In [ ]:
# Example predictions for new shipments
def predict_emission(vehicle_type, fuel_type, weight, distance):
    """Predict emissions for a new shipment"""
    vehicle_encoded = le_vehicle.transform([vehicle_type])[0]
    fuel_encoded = le_fuel.transform([fuel_type])[0]
    weight_ton = weight / 1000
    distance_weight_ratio = distance / weight
    
    features = np.array([[vehicle_encoded, fuel_encoded, weight, distance, 
                         weight_ton, distance_weight_ratio]])
    
    prediction = best_model.predict(features)[0]
    return prediction

# Example predictions
examples = [
    ('truck', 'electric', 2000, 500),
    ('truck', 'diesel', 2000, 500),
    ('plane', 'petrol', 1000, 1000),
    ('train', 'electric', 5000, 800),
    ('van', 'electric', 500, 200)
]

print("Example Emission Predictions:")
print("-" * 70)
for vehicle, fuel, weight, distance in examples:
    pred = predict_emission(vehicle, fuel, weight, distance)
    print(f"Vehicle: {vehicle:8s} | Fuel: {fuel:10s} | Weight: {weight:6.0f} kg | "
          f"Distance: {distance:5.0f} km | Predicted Emissions: {pred:7.2f} kg CO2")


## 8. Model Export and Recommendations


In [ ]:
import joblib
import json

# Save the best model
joblib.dump(best_model, 'emission_prediction_model.pkl')
joblib.dump(le_vehicle, 'vehicle_encoder.pkl')
joblib.dump(le_fuel, 'fuel_encoder.pkl')

# Save feature columns
with open('model_features.json', 'w') as f:
    json.dump(feature_cols, f)

print("Model saved successfully!")
print("\nRecommendations:")
print("1. Use electric vehicles for shorter distances to reduce emissions")
print("2. Trains are more efficient for heavy cargo over long distances")
print("3. Optimize route planning to minimize distance")
print("4. Consider hybrid vehicles as a middle ground between efficiency and flexibility")
print("5. Weight optimization can significantly reduce emissions per shipment")
